# Supplementary Results 4 — L2G against naive gene prioritisation

Supplementary Materials Table 1: sensitivity, specificity, precision and false discovery rate for
eight prioritisation rules, on the held-out set and on the whole gold standard. Supplementary
Table 12 reports the 2x2 tables behind the right-hand half.

Numbers are written to `results/sr04_l2g_vs_naive.json`. The five main-text values this section
supplies — the L2G recall of 0.65, the false discovery rates of 11.5% and 27%, and the eQTL
colocalisation sensitivity and FDR of 21.8% and 65.1% — are registered here under their Results 3
identifiers, since this is the only place they are computed.

**Provenance.** `manuscript_methods.l2g` holds the rules and the 2x2 helper, shared with
`chapters/06-supplementary-tables/03_l2g_tables.ipynb`.

**Two things about the published table's labels**, both checked in the last cell:

- its `L2G > 0.005` row is computed here as `L2G >= 0.05`, and the two are the *same rule* on this
  data: the prediction table is floored at 0.05, so no gene-CS pair carries a score between 0 and
  0.05 and every threshold in that interval selects the same pairs. The row is in effect "any gene
  the model scored at all".
- the thresholds are applied as `>=`, while the section states them as `>`. That makes no difference
  to the L2G or colocalisation rows, but it does to **PAV**: `vepMaximum > 0.66` gives sensitivity
  0.010 and FDR 0.083, nothing like the published 0.248 and 0.193, because the missense consequence
  score is exactly 0.66. The published row is `>= 0.66`, so the section's wording is wrong there.

In [1]:
import pandas as pd

from manuscript_methods import l2g, paper

numbers = {}
labelled = l2g.labelled_gold_standard()
print(f"gold-standard pairs: {len(labelled):,} | positive: {int(labelled['positive'].sum()):,}")
print(
    f"held out: {int(labelled['heldOut'].sum()):,} | positive among those: "
    f"{int(labelled.loc[labelled['heldOut'], 'positive'].sum()):,}"
)

/Users/yt4/Projects/Gentropy-manuscript/src/manuscript_methods/l2g.py:57: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


gold-standard pairs: 132,970 | positive: 8,520
held out: 18,611 | positive among those: 1,134


## The `Combined` rule

The manuscript's own prioritisation — every gene at L2G >= 0.5, plus the top-scoring gene of any
credible set with none above 0.5 — is materialised as `prioritised_genes_per_cs`, so it is read
rather than reimplemented. Recomputing it from raw scores misses the protein-coding restriction.

In [2]:
prioritised = pd.read_parquet(paper.derived("prioritised_genes_per_cs"), columns=["studyLocusId", "geneId"])
prioritised["prioritised"] = True
labelled = labelled.merge(prioritised, on=["studyLocusId", "geneId"], how="left")
labelled["prioritised"] = labelled["prioritised"].fillna(False).astype(bool)
print(f"gold-standard pairs the pipeline prioritises: {int(labelled['prioritised'].sum()):,}")

gold-standard pairs the pipeline prioritises: 8,177


/var/folders/p5/4t9crp1563l792qz8xz_3x5h0000gq/T/ipykernel_80122/4107641863.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


## The table

In [3]:
METRICS = ["Sensitivity (recall)", "Specificity (selectivity)", "PPV (precision)", "FDR"]
masks = l2g.evidence_masks(labelled, combined=labelled["prioritised"])

frames = {}
for universe, subset in [("test", labelled["heldOut"]), ("full", pd.Series(True, index=labelled.index))]:
    rows = [
        {**l2g.confusion(mask[subset], labelled.loc[subset, "positive"]), "Evidence": name}
        for name, mask in masks.items()
    ]
    frames[universe] = pd.DataFrame(rows).set_index("Evidence")

table = pd.concat(
    {"held-out set": frames["test"][METRICS], "training and test sets": frames["full"][METRICS]}, axis=1
).round(3)
table

held-out set                                            \
               Sensitivity (recall) Specificity (selectivity) PPV (precision)   
Evidence                                                                        
L2G>=0.5                      0.646                     0.995           0.885   
L2G>=0.05                     0.839                     0.932           0.443   
L2G>=0.8                      0.339                     0.998           0.923   
eQTL_coloc                    0.218                     0.974           0.349   
pQTL_coloc                    0.184                     0.992           0.613   
PAV                           0.248                     0.996           0.807   
Nearest to TSS                0.702                     0.983           0.730   
Combined                      0.776                     0.986           0.788   

                      training and test sets                            \
                  FDR   Sensitivity (recall) Specificity (selectivity)   
Evidence                                                                 
L2G>=0.5        0.115                  0.736                     0.995   
L2G>=0.05       0.557                  0.880                     0.948   
L2G>=0.8        0.077                  0.528                     0.999   
eQTL_coloc      0.651                  0.348                     0.960   
pQTL_coloc      0.387                  0.141                     0.997   
PAV             0.193                  0.207                     0.997   
Nearest to TSS  0.270                  0.644                     0.982   
Combined        0.212                  0.798                     0.989   

                                       
               PPV (precision)    FDR  
Evidence                               
L2G>=0.5                 0.904  0.096  
L2G>=0.05                0.538  0.462  
L2G>=0.8                 0.965  0.035  
eQTL_coloc               0.375  0.625  
pQTL_coloc               0.782  0.218  
PAV                      0.804  0.196  
Nearest to TSS           0.708  0.292  
Combined                 0.831  0.169

In [4]:
# (id, rule, universe, metric, published value) for every cell of the published table.
METRIC_COLUMN = dict(zip(["sensitivity", "specificity", "PPV", "FDR"], METRICS, strict=True))
PUBLISHED = [
    ("S4.01", "L2G>=0.5", "test", "sensitivity", 0.646),
    ("S4.02", "L2G>=0.5", "test", "specificity", 0.995),
    ("S4.03", "L2G>=0.5", "test", "PPV", 0.885),
    ("S4.04", "L2G>=0.5", "test", "FDR", 0.115),
    ("S4.05", "L2G>=0.5", "full", "sensitivity", 0.736),
    ("S4.06", "L2G>=0.5", "full", "specificity", 0.995),
    ("S4.07", "L2G>=0.5", "full", "PPV", 0.904),
    ("S4.08", "L2G>=0.5", "full", "FDR", 0.096),
    ("S4.09", "L2G>=0.05", "test", "sensitivity", 0.839),
    ("S4.10", "L2G>=0.05", "test", "specificity", 0.932),
    ("S4.11", "L2G>=0.05", "test", "PPV", 0.443),
    ("S4.12", "L2G>=0.05", "test", "FDR", 0.557),
    ("S4.13", "L2G>=0.05", "full", "sensitivity", 0.88),
    ("S4.14", "L2G>=0.05", "full", "specificity", 0.948),
    ("S4.15", "L2G>=0.05", "full", "PPV", 0.538),
    ("S4.16", "L2G>=0.05", "full", "FDR", 0.462),
    ("S4.17", "L2G>=0.8", "test", "sensitivity", 0.339),
    ("S4.18", "L2G>=0.8", "test", "specificity", 0.998),
    ("S4.19", "L2G>=0.8", "test", "PPV", 0.923),
    ("S4.20", "L2G>=0.8", "test", "FDR", 0.077),
    ("S4.21", "L2G>=0.8", "full", "sensitivity", 0.528),
    ("S4.22", "L2G>=0.8", "full", "specificity", 0.999),
    ("S4.23", "L2G>=0.8", "full", "PPV", 0.965),
    ("S4.24", "L2G>=0.8", "full", "FDR", 0.035),
    ("S4.25", "eQTL_coloc", "test", "sensitivity", 0.218),
    ("S4.26", "eQTL_coloc", "test", "specificity", 0.974),
    ("S4.27", "eQTL_coloc", "test", "PPV", 0.349),
    ("S4.28", "eQTL_coloc", "test", "FDR", 0.651),
    ("S4.29", "eQTL_coloc", "full", "sensitivity", 0.348),
    ("S4.30", "eQTL_coloc", "full", "specificity", 0.96),
    ("S4.31", "eQTL_coloc", "full", "PPV", 0.375),
    ("S4.32", "eQTL_coloc", "full", "FDR", 0.625),
    ("S4.33", "pQTL_coloc", "test", "sensitivity", 0.184),
    ("S4.34", "pQTL_coloc", "test", "specificity", 0.992),
    ("S4.35", "pQTL_coloc", "test", "PPV", 0.613),
    ("S4.36", "pQTL_coloc", "test", "FDR", 0.387),
    ("S4.37", "pQTL_coloc", "full", "sensitivity", 0.141),
    ("S4.38", "pQTL_coloc", "full", "specificity", 0.997),
    ("S4.39", "pQTL_coloc", "full", "PPV", 0.782),
    ("S4.40", "pQTL_coloc", "full", "FDR", 0.218),
    ("S4.41", "PAV", "test", "sensitivity", 0.248),
    ("S4.42", "PAV", "test", "specificity", 0.996),
    ("S4.43", "PAV", "test", "PPV", 0.807),
    ("S4.44", "PAV", "test", "FDR", 0.193),
    ("S4.45", "PAV", "full", "sensitivity", 0.207),
    ("S4.46", "PAV", "full", "specificity", 0.997),
    ("S4.47", "PAV", "full", "PPV", 0.804),
    ("S4.48", "PAV", "full", "FDR", 0.196),
    ("S4.49", "Nearest to TSS", "test", "sensitivity", 0.702),
    ("S4.50", "Nearest to TSS", "test", "specificity", 0.983),
    ("S4.51", "Nearest to TSS", "test", "PPV", 0.73),
    ("S4.52", "Nearest to TSS", "test", "FDR", 0.27),
    ("S4.53", "Nearest to TSS", "full", "sensitivity", 0.644),
    ("S4.54", "Nearest to TSS", "full", "specificity", 0.982),
    ("S4.55", "Nearest to TSS", "full", "PPV", 0.708),
    ("S4.56", "Nearest to TSS", "full", "FDR", 0.292),
    ("S4.57", "Combined", "test", "sensitivity", 0.776),
    ("S4.58", "Combined", "test", "specificity", 0.986),
    ("S4.59", "Combined", "test", "PPV", 0.788),
    ("S4.60", "Combined", "test", "FDR", 0.212),
    ("S4.61", "Combined", "full", "sensitivity", 0.798),
    ("S4.62", "Combined", "full", "specificity", 0.989),
    ("S4.63", "Combined", "full", "PPV", 0.831),
    ("S4.64", "Combined", "full", "FDR", 0.169),
]

for identifier, evidence, universe, metric, published in PUBLISHED:
    numbers[identifier] = round(float(frames[universe].loc[evidence, METRIC_COLUMN[metric]]), 3)

comparison = pd.DataFrame(
    [
        {
            "id": identifier,
            "rule": evidence,
            "universe": universe,
            "metric": metric,
            "published": published,
            "computed": numbers[identifier],
            "matches": abs(numbers[identifier] - published) <= 0.0015,
        }
        for identifier, evidence, universe, metric, published in PUBLISHED
    ]
)
print(f"cells reproducing the published table: {int(comparison['matches'].sum())} of {len(comparison)}")
comparison[~comparison["matches"]]

cells reproducing the published table: 64 of 64


,id,rule,universe,metric,published,computed,matches


## Main-text values this section supplies

In [5]:
numbers["R3.03"] = round(float(frames["test"].loc["L2G>=0.5", "Sensitivity (recall)"]), 2)
numbers["R3.06"] = round(100 * float(frames["test"].loc["L2G>=0.5", "FDR"]), 1)
numbers["R3.07"] = round(100 * float(frames["test"].loc["Nearest to TSS", "FDR"]), 0)
numbers["R3.13"] = round(100 * float(frames["test"].loc["eQTL_coloc", "Sensitivity (recall)"]), 1)
numbers["R3.14"] = round(100 * float(frames["test"].loc["eQTL_coloc", "FDR"]), 1)
print({k: numbers[k] for k in ["R3.03", "R3.06", "R3.07", "R3.13", "R3.14"]})

{'R3.03': 0.65, 'R3.06': 11.5, 'R3.07': 27.0, 'R3.13': 21.8, 'R3.14': 65.1}


## The two label checks

Neither changes a published cell; both are recorded so the definitions behind the table are explicit.

In [6]:
score = labelled["score"]
print(
    f"gene-CS pairs scored strictly between 0 and 0.05: {int(((score > 0) & (score < 0.05)).sum())} "
    f"(the prediction table is floored at {score[score > 0].min():.4f})"
)

strict = {
    "L2G > 0.5": score > 0.5,
    "L2G > 0.05": score > 0.05,
    "L2G > 0.005": score > 0.005,
    "L2G > 0.8": score > 0.8,
    "eQTL_coloc, strict": (labelled["eQtlColocClppMaximum"].fillna(0) > l2g.CLPP)
    | (labelled["eQtlColocH4Maximum"].fillna(0) > l2g.H4),
    "pQTL_coloc, strict": (labelled["pQtlColocClppMaximum"].fillna(0) > l2g.CLPP)
    | (labelled["pQtlColocH4Maximum"].fillna(0) > l2g.H4),
    "PAV, strict": labelled["vepMaximum"].fillna(0) > l2g.VEP_PAV,
}
held = labelled["heldOut"]
pd.DataFrame(
    [{**l2g.confusion(mask[held], labelled.loc[held, "positive"]), "Evidence": name} for name, mask in strict.items()]
).set_index("Evidence")[METRICS].round(3)

gene-CS pairs scored strictly between 0 and 0.05: 0 (the prediction table is floored at 0.0500)


,Sensitivity (recall),Specificity (selectivity),PPV (precision),FDR
Evidence,,,,
L2G > 0.5,0.646,0.995,0.885,0.115
L2G > 0.05,0.839,0.932,0.443,0.557
L2G > 0.005,0.839,0.932,0.443,0.557
L2G > 0.8,0.339,0.998,0.923,0.077
"eQTL_coloc, strict",0.218,0.974,0.349,0.651
"pQTL_coloc, strict",0.184,0.992,0.613,0.387
"PAV, strict",0.010,1.000,0.917,0.083


## Write the results

In [7]:
print(paper.save_results("sr04_l2g_vs_naive", numbers))
pd.Series(numbers).to_frame("computed")

/Users/yt4/Projects/Gentropy-manuscript/results/sr04_l2g_vs_naive.json


,computed
S4.01,0.646
S4.02,0.995
S4.03,0.885
S4.04,0.115
S4.05,0.736
...,...
R3.03,0.650
R3.06,11.500
R3.07,27.000
R3.13,21.800
